# 🚀 Fast Cloud Demo Benchmark (20 Dev + 20 Test)
### Detecting and Repairing Hallucinations in LLM-Generated Answers

This notebook runs the fast end-to-end cloud demo evaluation across:
- **20 Dev Records** (40 outputs: 20 B0 Baseline + 20 Guarded Pipeline)
- **20 Test Records** (40 outputs: 20 B0 Baseline + 20 Guarded Pipeline)
- **FEVER Wikipedia FTS5 Index** (reused directly from Google Drive)
- **Groq API** (`qwen/qwen3.8-27b`)

**Total runtime:** ~5 minutes with ZERO rate-limit issues!

## Step 1: Mount Google Drive & Sync Repository

In [ ]:
# Step 1: Mount Google Drive & Clone/Sync Repo
from google.colab import drive
import subprocess, sys, os
from pathlib import Path

# 1. Mount Google Drive
if not Path('/content/drive/MyDrive').exists():
    print("Mounting Google Drive...")
    drive.mount('/content/drive')
else:
    print("✓ Google Drive is already mounted.")

# 2. Sync Repository to latest main commit
REPO_URL = "https://github.com/harshitap1305/Detecting-and-Repairing-Hallucinations-in-LLM-Generated-Answers.git"
REPO = Path("/content/Detecting-and-Repairing-Hallucinations-in-LLM-Generated-Answers")

if not REPO.exists():
    print("Cloning repo...")
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
else:
    print("Syncing repo to origin/main...")
    subprocess.run(["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "reset", "--hard", "origin/main"], check=True)

print("✓ Repository synced to latest commit!")

## Step 2: Set Up Python 3.12 Isolated Virtualenv

In [ ]:
# Step 2: Set Up Isolated Python 3.12 Virtualenv
ENV = Path("/content/benchmark_env")
PYTHON312 = str(ENV / "bin/python")

if not Path(PYTHON312).exists():
    print("Installing uv and Python 3.12 venv...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "venv", "--python", "3.12", "--seed", str(ENV)], check=True)
    print("Installing requirements-evaluation.txt into venv...")
    subprocess.run([PYTHON312, "-m", "pip", "install", "-r", str(REPO / "requirements-evaluation.txt")], check=True)

version = subprocess.check_output([PYTHON312, "-c", "import sys; print(f'{sys.version_info.major}.{sys.version_info.minor}')"], text=True).strip()
print(f"✓ Isolated Python {version} environment ready: {PYTHON312}")

## Step 3: Link FEVER Wikipedia FTS5 Index from Google Drive

In [ ]:
# Step 3: Link FEVER SQLite Index from Drive
import shutil

INDEX = REPO / "indexes/fever_fts.sqlite"
INDEX.parent.mkdir(parents=True, exist_ok=True)

if not INDEX.exists() or INDEX.stat().st_size == 0:
    drive_candidates = [
        Path('/content/drive/MyDrive/hallucination_full_benchmark/fever_fts.sqlite'),
        Path('/content/drive/MyDrive/hallucination_reviewed_pilot/fever_fts.sqlite'),
    ]
    found = None
    for cand in drive_candidates:
        if cand.exists() and cand.stat().st_size > 1_000_000:
            found = cand
            break
    if not found:
        matches = list(Path('/content/drive/MyDrive').rglob('*fever_fts*.sqlite'))
        if matches:
            found = matches[0]

    if found:
        print(f"Found FEVER index in Drive: {found} ({found.stat().st_size / 1e9:.2f} GB)")
        if INDEX.exists() or INDEX.is_symlink():
            INDEX.unlink()
        INDEX.symlink_to(found)
    else:
        raise FileNotFoundError("fever_fts.sqlite not found in Google Drive! Ensure it is in MyDrive/hallucination_full_benchmark/")

print(f"✓ FEVER FTS5 Index ready at {INDEX} ({INDEX.stat().st_size / 1e9:.2f} GB)")

## Step 4: Configure & Verify Groq API Key

In [ ]:
# Step 4: Configure and Test Groq API Key
from google.colab import userdata

# 1. Try reading from Colab Secrets, else prompt/fallback
GROQ_KEY = None
try:
    GROQ_KEY = userdata.get("GROQ_API_KEY")
    print("✓ Found GROQ_API_KEY in Colab Secrets.")
except Exception:
    GROQ_KEY = os.environ.get("GROQ_API_KEY")

if not GROQ_KEY or not GROQ_KEY.startswith("gsk_"):
    # If secret is missing or blocked, paste directly here:
    GROQ_KEY = input("Enter your Groq API Key (starts with gsk_): ").strip()

os.environ["GROQ_API_KEY"] = GROQ_KEY

# 2. Test key with official groq client
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "groq"], check=True)
from groq import Groq

client = Groq(api_key=GROQ_KEY)
resp = client.chat.completions.create(
    model="qwen/qwen3.8-27b",
    messages=[{"role": "user", "content": "hi"}],
    max_tokens=2
)
print("✓ Groq API Key successfully verified working!")

## Step 5: Run Demo Benchmark on Dev Split (20 records, ~2.5 min)

In [ ]:
# Step 5: Run Demo Dev (40 outputs: 20 B0 Baseline + 20 Guarded Pipeline)
import json

DEV_OUT = REPO / "results/evaluation/demo_dev"
if DEV_OUT.exists() and not (DEV_OUT / "manifest.json").exists():
    shutil.rmtree(DEV_OUT)

cmd_dev = [
    PYTHON312, "-u", "scripts/run_benchmark.py",
    "--config", "configs/evaluation_demo.yaml",
    "--inputs", "data/gold/demo_dev.jsonl",
    "--splits", "data/demo_splits.json",
    "--output-dir", str(DEV_OUT),
    "--task", "repair",
    "--split", "dev",
    "--options-file", "configs/full_benchmark_v1.options.json",
    "--model", "qwen/qwen3.8-27b",
] + (["--resume"] if (DEV_OUT / "manifest.json").exists() else [])

env = os.environ.copy()
env["GROQ_API_KEY"] = GROQ_KEY

print(">>> Running Demo Dev (40 outputs, ~2.5 min)...")
subprocess.run(cmd_dev, check=True, cwd=REPO, env=env)
m_dev = json.loads((DEV_OUT / "manifest.json").read_text())
print(f"✓ Dev Complete: {m_dev['status']} ({m_dev['successful_outputs']}/{m_dev['expected_outputs']})")

## Step 6: Run Demo Benchmark on Test Split (20 records, ~2.5 min)

In [ ]:
# Step 6: Run Demo Test (40 outputs: 20 B0 Baseline + 20 Guarded Pipeline)
TEST_OUT = REPO / "results/evaluation/demo_test"
if TEST_OUT.exists() and not (TEST_OUT / "manifest.json").exists():
    shutil.rmtree(TEST_OUT)

cmd_test = [
    PYTHON312, "-u", "scripts/run_benchmark.py",
    "--config", "configs/evaluation_demo.yaml",
    "--inputs", "data/gold/demo_test.jsonl",
    "--splits", "data/demo_splits.json",
    "--output-dir", str(TEST_OUT),
    "--task", "repair",
    "--split", "test",
    "--options-file", "configs/full_benchmark_v1.options.json",
    "--model", "qwen/qwen3.8-27b",
] + (["--resume"] if (TEST_OUT / "manifest.json").exists() else [])

print(">>> Running Demo Test (40 outputs, ~2.5 min)...")
subprocess.run(cmd_test, check=True, cwd=REPO, env=env)
m_test = json.loads((TEST_OUT / "manifest.json").read_text())
print(f"✓ Test Complete: {m_test['status']} ({m_test['successful_outputs']}/{m_test['expected_outputs']})")

## Step 7: Generate Progress Reports & Backup to Google Drive

In [ ]:
# Step 7: Build Evaluation Progress Reports & Backup to Drive
import zipfile

print(">>> Generating evaluation report bundles...")
DRIVE_ROOT = Path('/content/drive/MyDrive/hallucination_full_benchmark')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

DEV_REP = REPO / "results/evaluation/report_demo_dev"
TEST_REP = REPO / "results/evaluation/report_demo_test"

subprocess.run([PYTHON312, "scripts/build_evaluation_report.py", "--run-dir", str(DEV_OUT),
                "--gold", "data/gold/demo_dev.jsonl", "--output-dir", str(DEV_REP), "--mode", "progress"], check=True, cwd=REPO)
subprocess.run([PYTHON312, "scripts/build_evaluation_report.py", "--run-dir", str(TEST_OUT),
                "--gold", "data/gold/demo_test.jsonl", "--output-dir", str(TEST_REP), "--mode", "progress"], check=True, cwd=REPO)

ZIP_DEST = DRIVE_ROOT / "demo_benchmark_results.zip"
with zipfile.ZipFile(ZIP_DEST, 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in [DEV_OUT, TEST_OUT, DEV_REP, TEST_REP]:
        for f in folder.rglob('*'):
            if f.is_file():
                zf.write(f, f.relative_to(REPO))

print(f"\n🎉 ALL DONE! Complete evaluation bundle archived to Google Drive:\n -> {ZIP_DEST}")